In [5]:
import pandas as pd
import joblib

inference = pd.read_csv(
    "../data/raw/inference.csv",
    parse_dates=["date"],
    dayfirst=True,
)
saved_random_forest = joblib.load("../models/random_forest.pkl")

print("Input rows:", len(inference))
print("Input columns:", inference.columns.tolist())
print("Date example:", inference["date"].iloc[0])
print("Model expects:", saved_random_forest.feature_names_in_.tolist())

Input rows: 71205
Input columns: ['index', 'store_ID', 'day_of_week', 'date', 'nb_customers_on_day', 'open', 'promotion', 'state_holiday', 'school_holiday']
Date example: 2015-03-01 00:00:00
Model expects: ['store_ID', 'day_of_week', 'open', 'promotion', 'school_holiday', 'month', 'day_of_month', 'week_of_year', 'state_holiday_0', 'state_holiday_a', 'state_holiday_b', 'state_holiday_c']


In [6]:
features = inference.copy()

features["month"] = features["date"].dt.month
features["day_of_month"] = features["date"].dt.day
features["week_of_year"] = features["date"].dt.isocalendar().week.astype(int)

features = pd.get_dummies(
    features,
    columns=["state_holiday"],
    dtype=int,
)

expected_columns = saved_random_forest.feature_names_in_.tolist()

print("Model columns missing before alignment:",
      sorted(set(expected_columns) - set(features.columns)))
print("Prepared rows:", len(features))

Model columns missing before alignment: []
Prepared rows: 71205


In [7]:
import numpy as np
from pathlib import Path

X_inference = features[expected_columns]

predicted_sales = np.maximum(
    saved_random_forest.predict(X_inference), 0
)
predicted_sales[inference["open"].to_numpy() == 0] = 0

# Reload without date conversion to preserve the original CSV columns and format.
prediction = pd.read_csv("../data/raw/inference.csv")
prediction["sales"] = np.rint(predicted_sales).astype(int)

output_path = Path("../data/Prediction/prediction.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
prediction.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(prediction))
display(prediction.head())

Saved: ..\data\Prediction\prediction.csv
Rows: 71205


,index,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,sales
0,272371,415,7,01/03/2015,0,0,0,0,0,0
1,558468,27,7,29/12/2013,0,0,0,0,0,0
2,76950,404,3,19/03/2014,657,1,1,0,0,4876
3,77556,683,2,29/01/2013,862,1,0,0,0,6640
4,456344,920,3,19/03/2014,591,1,1,0,0,5700
